# 📰  Pydantic 파이프라인 + 뉴스 모니터링

```
Phase 0~4. Core 학습 — Pydantic + RunnableParallel 핵심 부품
Project.   기사 → 분석 + 요약 + 태깅 + 대응 라우팅
```

## Phase 0. 환경 설정

In [1]:
%pip install -q langchain langchain-openai python-dotenv pydantic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: c:\Users\user\.pyenv\pyenv-win\versions\3.11.9\python.exe -m pip install --upgrade pip


In [2]:
# 1️⃣ 실행 환경 감지
try:
    import google.colab
    IS_COLAB = True
    print("✅ Google Colab")
except ImportError:
    IS_COLAB = False
    print("✅ 로컬 Jupyter")

# 3️⃣ API 키 로드
import os

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("✅ API 키 OK" if os.environ.get("OPENAI_API_KEY") else "❌ 키 없음")


from typing import Literal, Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

✅ 로컬 Jupyter
✅ API 키 OK


## Phase 1. PydanticOutputParser — 자연어를 객체로

5단계 패턴: 스키마 → 파서 → 프롬프트 → LLM → 체인

In [3]:
class Summary(BaseModel):
    title: str = Field(description="짧은 제목")
    sentiment: Literal["positive", "negative", "neutral"]
    key_points: list[str] = Field(description="핵심 2-3개")

parser = PydanticOutputParser(pydantic_object=Summary)

prompt = ChatPromptTemplate.from_messages([
    ("system", "텍스트를 분석합니다. 한국어로.\n\n{format_instructions}"),
    ("human", "{text}"),
]).partial(format_instructions=parser.get_format_instructions())

chain_summary = prompt | llm | parser

r = chain_summary.invoke({"text": "신제품 출시를 6월에서 7월로 연기하기로 했습니다."})
print(r)

title='신제품 출시 연기' sentiment='neutral' key_points=['신제품 출시 일정 변경', '출시가 6월에서 7월로 연기됨']


## Phase 2. RunnableParallel — 여러 체인 결합

In [4]:
# 자연어 답장 체인 추가 (StrOutputParser)
chain_notice = (
    ChatPromptTemplate.from_messages([
        ("system", "팀원에게 알릴 한 줄 안내문을 작성하세요."),
        ("human", "{text}"),
    ]) | llm | StrOutputParser()
)

combined = RunnableParallel(summary=chain_summary, notice=chain_notice)

r = combined.invoke({"text": "신제품 출시를 6월에서 7월로 연기하기로 했습니다."})
print(r["summary"])
print(r["notice"])

title='신제품 출시 연기' sentiment='neutral' key_points=['신제품 출시 일정 변경', '출시일이 6월에서 7월로 연기됨']
신제품 출시가 6월에서 7월로 연기되었음을 알려드립니다.


## Phase 3. RunnableLambda — 함수도 체인에 (결과는 Pydantic으로)

In [5]:
class Decision(BaseModel):
    urgent: bool
    note: str

def route(x) -> Decision:
    s = x["summary"]
    return Decision(urgent=(s.sentiment == "negative"), note=s.title)

chain_routed = combined | RunnableLambda(route)

d = chain_routed.invoke({"text": "신제품 출시 연기"})
print(d)

urgent=True note='신제품 출시 연기'


## Phase 4. RunnablePassthrough — 원본 보존

In [6]:
full = RunnableParallel(
    summary=chain_summary,
    notice=chain_notice,
    original=RunnablePassthrough(),
)
r = full.invoke({"text": "신제품 출시 연기"})
print(r["original"])  # 입력 그대로 통과

{'text': '신제품 출시 연기'}


---
# 📰 Project. 뉴스 모니터링 + PR 대응 시스템

```
기사 → 분석 + 임원 요약 + 태깅 (동시)
     → 대응 라우팅 (즉시 알림 / 일일 브리핑 / 보관)
```

**모니터링 대상**: NovaTech (가상 AI 스타트업, B2B 챗봇 솔루션)

In [ ]:
SAMPLE = """[테크크런치] NovaTech, B2B 챗봇 시장 점유율 1위 등극.
LLM 챗봇을 금융·제조 대기업 50여 곳에 공급하며 전년 대비 200% 성장.
다만 일부 고객사는 응답 속도 개선이 필요하다고 지적."""

# 분석 체인 — 자사 관점에서
class NewsAnalysis(BaseModel):
    headline: str = Field(description="15자 이내 헤드라인")
    sentiment: Literal["positive", "negative", "mixed", "neutral"]
    impact_score: int = Field(ge=0, le=10, description="자사 영향도 0-10")
    contains_risk: bool = Field(description="평판/사업 리스크 언급 여부")

p1 = PydanticOutputParser(pydantic_object=NewsAnalysis)
chain_analysis = (
    ChatPromptTemplate.from_messages([
        ("system", "자사(NovaTech, B2B AI 챗봇 스타트업) 관점에서 기사를 분석합니다.\n\n{format_instructions}"),
        ("human", "{article}"),
    ]).partial(format_instructions=p1.get_format_instructions())
    | llm | p1
)
print(chain_analysis.invoke({"article": SAMPLE}))

headline='NovaTech, B2B 챗봇 시장 1위' sentiment='positive' impact_score=8 contains_risk=True


In [8]:
# 임원 보고용 3문장 요약 (자연어)
chain_brief = (
    ChatPromptTemplate.from_messages([
        ("system", "임원 보고용 *정확히 3문장* 요약: 사실 / 자사 관련성 / 시사점."),
        ("human", "{article}"),
    ]) | llm | StrOutputParser()
)
print(chain_brief.invoke({"article": SAMPLE}))

NovaTech는 B2B 챗봇 시장에서 1위로 등극하며, LLM 챗봇을 50여 개 금융 및 제조 대기업에 공급하여 전년 대비 200% 성장했다. 자사와 관련하여, 이러한 성장은 고객의 요구에 부응하는 혁신적인 솔루션 제공을 통해 이루어졌음을 보여준다. 그러나 응답 속도 개선이 필요하다는 고객의 피드백은 향후 서비스 개선 및 경쟁력 강화를 위한 중요한 시사점을 제공한다.


In [9]:
# 태깅 체인 — 다중 선택 (list[Literal])
class NewsTags(BaseModel):
    topics: list[Literal["제품", "재무", "인사", "기술", "규제", "경쟁사", "기타"]]
    teams: list[Literal["PR", "마케팅", "프로덕트", "법무", "재무"]] = Field(
        description="이 기사를 봐야 할 사내 팀"
    )

p2 = PydanticOutputParser(pydantic_object=NewsTags)
chain_tags = (
    ChatPromptTemplate.from_messages([
        ("system", "기사에 적절한 주제와 관련 팀 태그.\n\n{format_instructions}"),
        ("human", "{article}"),
    ]).partial(format_instructions=p2.get_format_instructions())
    | llm | p2
)

# 3개 체인 동시 실행
monitor_v1 = RunnableParallel(
    analysis=chain_analysis,
    brief=chain_brief,
    tags=chain_tags,
)

r = monitor_v1.invoke({"article": SAMPLE})
print(f"📌 {r['analysis'].headline} | 영향도 {r['analysis'].impact_score}")
print(f"🏷  {r['tags'].topics} / 팀: {r['tags'].teams}")

📌 NovaTech, B2B 챗봇 시장 1위 | 영향도 8
🏷  ['기술', '경쟁사'] / 팀: ['프로덕트', '마케팅']


In [10]:
# 대응 라우팅 — RunnableLambda + Pydantic
class PRAction(BaseModel):
    priority: Literal["alert_now", "daily_brief", "archive"]
    channel: Literal["slack_urgent", "slack_daily", "notion"]
    message: str

def route_pr(x) -> PRAction:
    a = x["analysis"]
    if a.sentiment == "negative" and a.impact_score >= 7:
        return PRAction(priority="alert_now", channel="slack_urgent",
                       message=f"🚨 즉시 대응: {a.headline}")
    if a.impact_score >= 4:
        return PRAction(priority="daily_brief", channel="slack_daily",
                       message=f"📋 브리핑: {a.headline}")
    return PRAction(priority="archive", channel="notion",
                   message=f"📁 보관: {a.headline}")

monitor_v2 = monitor_v1 | RunnableLambda(route_pr)
print(monitor_v2.invoke({"article": SAMPLE}))

priority='daily_brief' channel='slack_daily' message='📋 브리핑: NovaTech, B2B 챗봇 시장 1위'


In [11]:
# 여러 기사 batch — 매일 들어오는 수십 건
ARTICLES = [
    {"article": SAMPLE},
    {"article": "NovaTech 일부 고객사 데이터 유출 의혹. 회사는 부인. 보안 업계 우려 제기."},
    {"article": "정부, AI 산업 5조원 지원금 편성. 스타트업·연구기관 대상."},
]
for a in monitor_v2.batch(ARTICLES):
    icon = {"alert_now": "🚨", "daily_brief": "📋", "archive": "📁"}[a.priority]
    print(f"{icon} {a.message}")

📋 📋 브리핑: NovaTech, B2B 챗봇 시장 1위
🚨 🚨 즉시 대응: 고객사 데이터 유출 의혹
📋 📋 브리핑: 정부, AI 산업 지원


---
## 🎓 마무리

같은 3개 부품으로 PR 모니터링 시스템 완성. **도메인이 달라도 패턴은 동일**.

**도전과제**: SNS 카피 체인 / 영문 번역 체인 / 경쟁사 비교 체인 추가 중 택1

In [29]:
SAMPLE = """[테크크런치] NovaTech, B2B 챗봇 시장 점유율 1위 등극.
LLM 챗봇을 금융·제조 대기업 50여 곳에 공급하며 전년 대비 200% 성장.
다만 일부 고객사는 응답 속도 개선이 필요하다고 지적."""

SNS_SYSTEM = """너는 SNS(인스타그램, 스레드)에서 바이럴과 알고리즘 확산을 전문으로 하는 '탑티어 SNS 카피라이터'이자 '퍼포먼스 마케터'야.
유저가 콘텐츠를 보자마자 멈추고, 본문을 끝까지 읽은 뒤, 결국 '저장'과 '공유(태그)'를 누르게 만드는 '카피 체인(Copy Chain)' 구조의 문장을 작성해 줘.

# 작성 가이드라인
아래 4단계 구조로 서로 다른 버전 3개를 작성해.
1. [Hooking] 시선 포착 헤드카피 1~2줄: 타겟의 문제를 찌르거나 호기심 극대화
2. [Body] 본문 요약: 이모지·리스트로 짧고 가독성 좋게
3. [CTA 1] 저장 유도: 나중에 다시 봐야 할 이유 제시
4. [CTA 2] 공유/태그 유도: 친구·동료를 소환할 명분 제공

# SNS 카피 체인
class Summary(BaseModel):
    title: str = Field(description="짧은 제목")
    sentiment: Literal["positive", "negative", "neutral"]
    key_points: list[str] = Field(description="핵심 2-3개")"""

parser = PydanticOutputParser(pydantic_object=Summary)

prompt = ChatPromptTemplate.from_messages([
    ("system", "텍스트를 분석합니다. 한국어로.\n\n{format_instructions}"),
    ("human", "{text}"),
]).partial(format_instructions=parser.get_format_instructions())



#==================================================================================
# # 1. 스키마 작성
# class ContentAnalysis(BaseModel):
#     product: str = Field(description="홍보할 제품/서비스/주제 한 줄")
#     target: str = Field(description="가장 반응할 핵심 타겟 (예: 2030 직장인, B2B 도입 담당자)")
#     value: str = Field(description="타겟에게 줄 핵심 가치 한 줄")
#     tone: Literal["친근", "전문가", "유머", "감성"] = Field(description="이 콘텐츠에 가장 맞는 톤")
#     viral_score: int = Field(ge=0, le=10, description="SNS 확산 가능성. 10=저장·공유 폭발, 5=보통, 0=반응 없음")
#     risk: Literal["낮음", "중간", "높음"] = Field(description="부정 이슈·논란·법적 리스크 수준")
# # 2. 파서
# analysis_parser = PydanticOutputParser(pydantic_object=ContentAnalysis)
# # 3. 프롬프트작성
# analysis_prompt = ChatPromptTemplate.from_messages([
#     ("system", "너는 SNS 퍼포먼스 마케터야. 원문을 SNS 홍보 관점에서 분석해.\n"
#                "tone과 risk는 반드시 주어진 보기 중 하나로만 답해.\n\n{format_instructions}"),
#     ("human", "{text}"),
# ]).partial(format_instructions=analysis_parser.get_format_instructions())
# # 4. llm=> 앞에서 진행함.
# # 5. 체인
# chain_analysis = analysis_prompt | llm | analysis_parser

# r = chain_analysis.invoke({'text':SAMPLE})

# print(r)

#===================================================================================
# SNS 카피
#1.스키마
class CopySet(BaseModel):
    hooking: str = Field(description="시선 포착 헤드카피 1~2줄. 타겟의 문제를 찌르거나 호기심 극대화")
    body: str = Field(description="핵심 가치를 이모지·리스트로 짧게 요약한 본문")
    cta_save: str = Field(description="나중에 다시 봐야 할 이유를 제시하는 저장 유도 카피")
    cta_share: str = Field(description="친구·동료를 소환할 명분을 주는 공유/태그 유도 카피")

class CopyChain(BaseModel):
    versions: list[CopySet] = Field(description="서로 다른 접근의 카피 세트 3개", min_length=3, max_length=3)
#2. 파서
copy_parser = PydanticOutputParser(pydantic_object=CopyChain)
#3. 프롬프트
SNS_SYSTEM = """너는 SNS(인스타그램, 스레드)에서 바이럴과 알고리즘 확산을 전문으로 하는 '탑티어 SNS 카피라이터'이자 '퍼포먼스 마케터'야.
유저가 보자마자 멈추고, 본문을 끝까지 읽은 뒤, 결국 '저장'과 '공유(태그)'를 누르게 만드는 '카피 체인' 3개 버전을 작성해.
세 버전은 접근 방식이 서로 달라야 해 (예: 문제 제기형 / 숫자·성과형 / 질문·호기심형).
부정적인 내용은 카피에 넣지 마."""

copy_prompt = ChatPromptTemplate.from_messages([
    ("system", SNS_SYSTEM + "\n\n{format_instructions}"),
    ("human", "{text}"),
]).partial(format_instructions=copy_parser.get_format_instructions())
#4.체인
chain_copy = copy_prompt|llm|copy_parser

r = chain_copy.invoke({"text": SAMPLE})

for i, v in enumerate(r.versions, 1):
    print(f"\n── 버전 {i} ──\n[Hooking] {v.hooking}\n[Body] {v.body}\n[CTA 1] {v.cta_save}\n[CTA 2] {v.cta_share}")


── 버전 1 ──
[Hooking] B2B 챗봇 시장의 판도를 바꾼 NovaTech! 🚀
[Body] ✅ 1위 점유율 달성!  
✅ 50개 대기업에 공급!  
✅ 전년 대비 200% 성장!  

이제 챗봇의 시대가 열렸습니다!
[CTA 1] 이 놀라운 성장을 놓치지 마세요! 저장해두고 나중에 다시 확인하세요.
[CTA 2] 이 정보를 친구와 공유하고, 함께 챗봇의 미래를 논의해보세요! 👥

── 버전 2 ──
[Hooking] 챗봇 시장의 새로운 강자, NovaTech! 📈
[Body] 💡 200% 성장의 비결은?  
💡 50개 대기업과의 파트너십!  
💡 응답 속도 개선이 필요하다는 피드백도!
[CTA 1] 이 성장 스토리를 저장해두고, 나중에 더 많은 인사이트를 얻어보세요.
[CTA 2] 이 기사를 친구에게 태그하고, 함께 챗봇의 혁신을 이야기해보세요!

── 버전 3 ──
[Hooking] NovaTech의 성공 비결, 궁금하지 않으세요? 🤔
[Body] 🌟 B2B 챗봇 시장 1위!  
🌟 200% 성장!  
🌟 50개 대기업과의 협력!
[CTA 1] 이 성공 사례를 저장해두고, 나중에 참고하세요!
[CTA 2] 이 흥미로운 이야기를 친구와 나누고, 함께 챗봇의 미래를 탐구해보세요!


### 처음 진행했었던

In [23]:
chain_sns = (
    ChatPromptTemplate.from_messages([
        ("system", SNS_SYSTEM),
        ("human", "{text}"),
    ]) | llm | StrOutputParser()
)
# print(chain_sns.invoke({"text": SAMPLE}))

combine_sns = RunnableParallel(summary=chain_summary,
                                 notice=chain_notice,
                                 SNS_copy = chain_sns)

r = combine_sns.invoke({"text":SAMPLE})
print(r["SNS_copy"])

### 카피 체인 버전 1: 문제 제기형
"챗봇, 정말 필요한가요? 🤔  
B2B 시장에서 NovaTech가 1위에 올랐다는 소식! 금융과 제조 대기업 50곳에 공급하며 무려 200% 성장! 🚀  
하지만, 응답 속도에 대한 고객의 목소리도 잊지 마세요. 여러분의 비즈니스에 최적의 챗봇은 무엇일까요? 💡  
👉 이 포스트를 저장하고, 친구와 함께 고민해보세요!"

---

### 카피 체인 버전 2: 숫자·성과형
"📈 NovaTech, B2B 챗봇 시장의 새로운 강자!  
- 1위 점유율 달성  
- 50개 대기업에 공급  
- 전년 대비 200% 성장!  
이 놀라운 성과의 비결은 무엇일까요? 🤔  
지금 바로 확인하고, 이 정보를 친구와 공유하세요! 🔗✨"

---

### 카피 체인 버전 3: 질문·호기심형
"챗봇이 비즈니스를 어떻게 변화시킬까요? 💬  
NovaTech가 B2B 챗봇 시장에서 1위에 올랐다는 사실, 알고 계셨나요?  
50여 개 대기업에 공급하며 200% 성장한 그 비결은?  
여러분의 비즈니스에도 챗봇이 필요할까요? 🤔  
이 포스트를 저장하고, 여러분의 생각을 태그해보세요! 🏷️"
